This week we will be learning about aggregations and using them to do a deeper investigation into the effects of different types of joins. 

We will also do a few problems exploring matrix inverses. 

### Tutorials
- https://www.datacamp.com/community/tutorials/pandas-multi-index
- https://www.datacamp.com/community/tutorials/pandas-split-apply-combine-groupby

## Grouping Data

The expressiveness of Python and pandas allows complex group operations using any function that accepts a pandas object or NumPy array. This can include:

- Splitting a pandas object into pieces using one or more keys
- Calculating group summary statistics
- Applying within-group transformations or other manipulations
- Computing pivot tables and cross-tabulations
- Performing quantile analysis and other statistical group analyses

### GroupBy Operations

Group operations involve the `split-apply-combine` mechanism.

1. Data are split into groups based on one or more keys
2. A function is applied to each group
3. Results of the function application are combined into a new object

Grouping keys can take many forms, and the keys do not have to be all of the same type.

pandas `groupby` method returns a GroupBy object that can be re-used.

DataFrame columns can be used as the group keys.

Numeric aggregations will exclude `nuisance` (non-numeric) columns from the result

By default `groupby` groups on axis=0, but can group on any of the other axes.

### Iterating over groups

The GroupBy object supports iteration, generating a sequence of 2-tuples containing the group name along with the chunk of data.

Indexing a GroupBy object created from a DataFrame with a column name or array of column names has the effect of column subsetting for aggregation. This means that:
```
df.groupby('key1')['data1']
```
is essentially the same as:
```
df['data1'].groupby(df['key1'])
```

### Reindexing

The output of GroupBys are often multi-indexed. This is typically not the form you want for your analysis. This means that you will want to reset your index or remove multi indexes on your columns. See below for examples of that. 

In [1]:
import pandas as pd
import numpy as np

speeds = pd.DataFrame(
    [
        ("bird", "Falconiformes", 389.0, 0),
        ("bird", "Psittaciformes", 24.0, 0),
        ("mammal", "Carnivora", 80.2, 0),
        ("mammal", "Primates", np.nan, 0),
        ("mammal", "Carnivora", 58, 0),
    ],
    index=["falcon", "parrot", "lion", "monkey", "leopard"],
    columns=("class", "order", "max_speed", 'min_speed'),
)

not_reindexed = speeds.groupby(["class", "order"]).agg(['sum', 'mean'])
not_reindexed

max_speed        min_speed     
                            sum   mean       sum mean
class  order                                         
bird   Falconiformes      389.0  389.0         0  0.0
       Psittaciformes      24.0   24.0         0  0.0
mammal Carnivora          138.2   69.1         0  0.0
       Primates             0.0    NaN         0  0.0

In [2]:
not_reindexed.columns = ["_".join(i) for i in not_reindexed.columns]
not_reindexed

max_speed_sum  max_speed_mean  min_speed_sum  \
class  order                                                          
bird   Falconiformes           389.0           389.0              0   
       Psittaciformes           24.0            24.0              0   
mammal Carnivora               138.2            69.1              0   
       Primates                  0.0             NaN              0   

                       min_speed_mean  
class  order                           
bird   Falconiformes              0.0  
       Psittaciformes             0.0  
mammal Carnivora                  0.0  
       Primates                   0.0

In [3]:
not_reindexed.reset_index()

,class,order,max_speed_sum,max_speed_mean,min_speed_sum,min_speed_mean
0,bird,Falconiformes,389.0,389.0,0,0.0
1,bird,Psittaciformes,24.0,24.0,0,0.0
2,mammal,Carnivora,138.2,69.1,0,0.0
3,mammal,Primates,0.0,NaN,0,0.0


### All Data Descriptions


* `customers` - A table of customers. The unique id for the customer is `customer_id` meaning only one id per customer. 
* `orders` - A table of orders. The unique id for the order is `order_id` there is a foreign key (`customer_id`) that tells you which customer placed the order.
* `web_visits`: Describes visitors to the website, with `visitor_id` be a unique id per visitor to the website. If the visitor is a logged in customer then the `customer_id` has a value otherwise it is null. 
* `ad_clicks`: Describes clicks on ad on an external website, the `external_user_id` is an unique id per user from the external website. If the user is a known in customer then the `customer_id` has a value otherwise it is null. 

In [4]:
import generate_week_10_data

customers, orders, web_visits, ad_clicks = generate_week_10_data.create_data()

### Problem 1 (10 pts)

1. What is the order amount per `customer_name`, only include customers who have orders
2. What is the order amount per `customer_name`, include customers even if they have no orders

#### Grading: 
5 points per question. 2 points for the correct join and 3 point for the aggragations. 

In [9]:
# Only customers who have orders (inner join)
inner = customers.merge(orders, on="customer_id", how="inner")
# For each customer_name, sum the order_amount and return a dataFrame
order_amount_inner = inner.groupby("customer_name")["order_amount"].sum().reset_index()
order_amount_inner.columns = ["customer_name", "total_order_amount"]
print("\n Question 1")
display(order_amount_inner)

# Include customers even if they have no orders (left join)
left = customers.merge(orders, on="customer_id", how="left")
# For each customer_name, sum the order_amount and return a dataFrame
order_amount_left = left.groupby("customer_name")["order_amount"].sum().reset_index()
order_amount_left.columns = ["customer_name", "total_order_amount"]
print("\n Question 2")
display(order_amount_left)


 Question 1


,customer_name,total_order_amount
0,amputator,392.529244
1,chyloid,1648.153348
2,dermatoma,997.920841
3,digitogenin,709.626087
4,lamnectomy,1583.329018
5,rapscallionly,997.713906
6,receivership,1215.266089
7,shita,866.183315
8,tasselet,547.586156



 Question 2


,customer_name,total_order_amount
0,amputator,392.529244
1,chyloid,1648.153348
2,dermatoma,997.920841
3,digitogenin,709.626087
4,lamnectomy,1583.329018
5,rapscallionly,997.713906
6,receivership,1215.266089
7,septal,0.000000
8,shita,866.183315
9,tasselet,547.586156


### Problem 2 (5 pts)
If you look at the raw results of the join you can visually see that there are duplicates for the columns that started in the `customer` table. This wasn't a concern above because we we doing an aggregation on a column in the `orders` table. Let's think through problems that require a join but are looking to do an aggregation with a column from the `customers` table. 

1. What is the average `customer_age` for customers who have placed orders. For this problem print the accurate average `customer_age` with duplicates remove and also print the inaccure average `customer_age` without duplicates removed. 
2. What is the average `customer_age` for customers who have placed orders versus those who haven't. 

#### Grading: 
5 points per question. 2 points for the correct join and 3 point for the aggragations. 

In [31]:
# Problem 1: Average customer_age for customers who have placed orders
# Inner join to get only customers with orders
inner = customers.merge(orders, on="customer_id", how="inner")

# Average customer_age without removing duplicates
inaccurate_avg = inner["customer_age"].mean()
print(f"Inaccurate average customer_age (with duplicates): {inaccurate_avg:.2f}")

# Average customer_age with duplicates removed
# drop_duplicates on customer_id so each customer is counted once
accurate_avg = inner.drop_duplicates(subset="customer_id")["customer_age"].mean()
print(f"Accurate average customer_age (duplicates removed): {accurate_avg:.2f}")

# Problem 2: Average customer_age for customers who have placed orders vs those who haven't
# Left join so all customers are included
left = customers.merge(orders, on="customer_id", how="left")

# Has_orders is True if order_id is not null
left["has_orders"] = left["order_id"].notna()

# Group by has_orders and calculate average customer_age for each group
avg_by_status = (left.drop_duplicates(subset="customer_id")
                     .groupby("has_orders")["customer_age"]
                     .mean()
                     .reset_index())
avg_by_status.columns = ["has_orders", "avg_customer_age"]
display(avg_by_status)

Inaccurate average customer_age (with duplicates): 11.24
Accurate average customer_age (duplicates removed): 13.37


,has_orders,avg_customer_age
0,False,27.767543
1,True,13.370208


### Problem 3 

Per customer what is the total number of web visits and ads clicked on? 

For this problem I want you to start with the `customers_visits_clicks_raw` table below and again do an accurate count that only counts a web visit once and an inaccurate count that counts duplications. Here is what the results should look like: 

<img src="images/number_visits_clicks.png" width="800"/>


#### Grading: 
5 points for joining correctly. 5 points for the correct aggragtions.  


In [ ]:
# Start with this dataset (already joined: customers + web_visits + ad_clicks)
customers_visits_clicks_raw = customers.merge(web_visits, on="customer_id", how="left").merge(ad_clicks, on="customer_id", how="left")

# Inaccurate count because of duplicated rows from the many-to-many join
# agg counting interaction_id and click_id will overcount because of duplicates from the join
inaccurate = customers_visits_clicks_raw.groupby("customer_name").agg(
    num_web_visits=("interaction_id", "count"),   
    num_ad_clicks=("click_id", "count")           
).reset_index()

print("Inaccurate count (with duplicates from the join):")
display(inaccurate)

# Accurate count
# Don't count duplicates in the aggregation by using nunique to count unique interaction_id and click_id for each customer_name
accurate = customers_visits_clicks_raw.groupby("customer_name").agg(
    num_web_visits=("interaction_id", "nunique"),  
    num_ad_clicks=("click_id", "nunique")          
).reset_index()

print("\nAccurate count (duplicates removed with nunique):")
display(accurate)

Inaccurate count (with duplicates from the join):


,customer_name,num_web_visits,num_ad_clicks
0,amputator,0,3
1,chyloid,0,2
2,dermatoma,0,1
3,digitogenin,0,1
4,lamnectomy,1,1
5,rapscallionly,3,3
6,receivership,2,0
7,septal,2,2
8,shita,1,1
9,tasselet,1,0



Accurate count (duplicates removed with nunique):


,customer_name,num_web_visits,num_ad_clicks
0,amputator,0,3
1,chyloid,0,2
2,dermatoma,0,1
3,digitogenin,0,1
4,lamnectomy,1,1
5,rapscallionly,1,3
6,receivership,2,0
7,septal,2,1
8,shita,1,1
9,tasselet,1,0


### Problem 4 (5 pts)

We will now work through a problem with outer joins.

Create a dataframe with the daily number of: 
* web visits
* unique web visitors
* unique customers visiting website
* ads clicks
* unique users clicking
* unique customers clicking

You should include the full date range 2024-05-01 to 2024-05-09 even though not all dates are included in either dataframe. 

You will create this data frame two ways:
1. Join `web_visits` to `ad_clicks` first and then perform the calculations
2. Create aggregate dataframes for `web_visits` to `ad_clicks` first and then join those aggregate dataframes together

#### Grading: 
5 points per method. 2 points for the correct join and 3 point for the aggragations. 

In [30]:
# Problem 1: join first, then aggregate
# Outer join on date for all dates for both
# Suffixes are needed to tell apart customer_id columns
joined = web_visits.merge(ad_clicks, on="date", how="outer", suffixes=("_web", "_ad"))

# Group by date and compute the six daily metrics
method1 = joined.groupby("date").agg(
    web_visits=("interaction_id", "count"),               
    unique_web_visitors=("visitor_id", "nunique"),         
    unique_web_customers=("customer_id_web", "nunique"),   
    ad_clicks=("click_id", "count"),                       
    unique_ad_users=("external_user_id", "nunique"),       
    unique_ad_customers=("customer_id_ad", "nunique")      
).reset_index()

# Reindex for full date range 
full_dates = pd.date_range("2024-05-01", "2024-05-09")
method1 = method1.set_index("date").reindex(full_dates, fill_value=0).reset_index()
method1.rename(columns={"index": "date"}, inplace=True)

print("Method 1: join first then aggregate:")
display(method1)


# Problem 2: aggregate first, then join the two summary tables
# Summarise web_visits by date
web_agg = web_visits.groupby("date").agg(
    web_visits=("interaction_id", "count"),
    unique_web_visitors=("visitor_id", "nunique"),
    unique_web_customers=("customer_id", "nunique")
).reset_index()

# Summarise ad_clicks by date
ad_agg = ad_clicks.groupby("date").agg(
    ad_clicks=("click_id", "count"),
    unique_ad_users=("external_user_id", "nunique"),
    unique_ad_customers=("customer_id", "nunique")
).reset_index()

# Outer join the two summaries so all dates from both appear
method2 = web_agg.merge(ad_agg, on="date", how="outer")

# Reindex to the full date range and fill missing values with 0
method2 = method2.set_index("date").reindex(full_dates, fill_value=0).reset_index()
method2.rename(columns={"index": "date"}, inplace=True)

print("\nMethod 2: aggregate first then join:")
display(method2)

Method 1: join first then aggregate:


,date,web_visits,unique_web_visitors,unique_web_customers,ad_clicks,unique_ad_users,unique_ad_customers
0,2024-05-01,4,4,1,0,0,0
1,2024-05-02,5,5,2,0,0,0
2,2024-05-03,10,5,3,10,1,1
3,2024-05-04,15,5,2,15,3,3
4,2024-05-05,3,1,0,3,3,2
5,2024-05-06,0,0,0,3,3,2
6,2024-05-07,0,0,0,3,3,2
7,2024-05-08,0,0,0,3,3,2
8,2024-05-09,0,0,0,3,3,0



Method 2: aggregate first then join:


,date,web_visits,unique_web_visitors,unique_web_customers,ad_clicks,unique_ad_users,unique_ad_customers
0,2024-05-01,4.0,4.0,1.0,NaN,NaN,NaN
1,2024-05-02,5.0,5.0,2.0,NaN,NaN,NaN
2,2024-05-03,5.0,5.0,3.0,2.0,1.0,1.0
3,2024-05-04,5.0,5.0,2.0,3.0,3.0,3.0
4,2024-05-05,1.0,1.0,0.0,3.0,3.0,2.0
5,2024-05-06,NaN,NaN,NaN,3.0,3.0,2.0
6,2024-05-07,NaN,NaN,NaN,3.0,3.0,2.0
7,2024-05-08,NaN,NaN,NaN,3.0,3.0,2.0
8,2024-05-09,NaN,NaN,NaN,3.0,3.0,0.0


### Problem 5 (15 pts)

Using the same two methods as problem 4 attempt to include:
* mean time on page per date 
* total time on page per date

One method will lead to some incorrect values unless you do a bit more work to correct the values. 

Write out a which method leads to correct values, which leads to incorrect values and why. 

#### Grading: 
* 5 points per method. 2 points for the correct join and 3 point for the aggragations. 
* 5 points for the written explaination. 

In [27]:
# Problem 1: Join first, then aggregate
# Outer join web_visits and ad_clicks on date
joined = web_visits.merge(ad_clicks, on="date", how="outer", suffixes=("_web", "_ad"))

# Create dataFrame 1
method1 = joined.groupby("date").agg(
    mean_time_on_page=("time_on_page", "mean"),
    total_time_on_page=("time_on_page", "sum")
).reset_index()

# Fill in the full date range
full_dates = pd.date_range("2024-05-01", "2024-05-09")
method1 = method1.set_index("date").reindex(full_dates).reset_index()
method1.rename(columns={"index": "date"}, inplace=True)

print("Method 1: join first then aggregate (incorrect)")
display(method1)


# Problem 2: Aggregate first, then join 
# Create dataFrame 2
web_time_agg = web_visits.groupby("date").agg(
    mean_time_on_page=("time_on_page", "mean"),
    total_time_on_page=("time_on_page", "sum")
).reset_index()

# Join with an ad_clicks for full date range
ad_agg = ad_clicks.groupby("date").size().reset_index(name="ad_click_count")
method2 = web_time_agg.merge(ad_agg, on="date", how="outer").drop(columns=["ad_click_count"])

# Fill in full date range
method2 = method2.set_index("date").reindex(full_dates).reset_index()
method2.rename(columns={"index": "date"}, inplace=True)

print("\nMethod 2: aggregate first then join (correct):")
display(method2)

# Explanation 
"""
Method 1 give you incorrect values 
Mathod 2 give you correct values 
When you outer-join web_visits and ad_clicks on date berfore aggregating,
every web_visit row on a shared date gets duplicated, once for each ad_click
on that same date. Method 2 avoids this by computing the aggregation on 
web_visits first so each time_on_page value is counted once.
"""

Method 1: join first then aggregate (incorrect)


,date,mean_time_on_page,total_time_on_page
0,2024-05-01,38.955214,155.820856
1,2024-05-02,53.381828,266.909141
2,2024-05-03,60.249733,602.497325
3,2024-05-04,22.672504,340.087556
4,2024-05-05,55.757819,167.273457
5,2024-05-06,NaN,0.000000
6,2024-05-07,NaN,0.000000
7,2024-05-08,NaN,0.000000
8,2024-05-09,NaN,0.000000



Method 2: aggregate first then join (correct):


,date,mean_time_on_page,total_time_on_page
0,2024-05-01,38.955214,155.820856
1,2024-05-02,53.381828,266.909141
2,2024-05-03,60.249733,301.248663
3,2024-05-04,22.672504,113.362519
4,2024-05-05,55.757819,55.757819
5,2024-05-06,NaN,NaN
6,2024-05-07,NaN,NaN
7,2024-05-08,NaN,NaN
8,2024-05-09,NaN,NaN


'\nMethod 1 give you incorrect values \nMathod 2 give you correct values \nWhen you outer-join web_visits and ad_clicks on date berfore aggregating,\nevery web_visit row on a shared date gets duplicated, once for each ad_click\non that same date. Method 2 avoids this by computing the aggregation on \nweb_visits first so each time_on_page value is counted once.\n'

### Problem 6 (15 pts)

Implement the full algorithm described in [Inverting Any Square Full-Rank Matrix](https://learning.oreilly.com/library/view/practical-linear-algebra/9781098120603/ch08.html#inverting-any-square) and reproduce Figure 8-3. Of course, your matrices will look different from Figure 8-3 because of random numbers, although the grid and identity matrices will be the same.

#### Grading: 
* 10 points for implimenting the algorithm. 
* 5 points for reproducing the figure. 

### Problem 7 (15 pts)

The LIVE EVIL rule applies to the inverse of multiplied matrices. Test this in code by creating two square full-rank matrices $A$ and $B$, then use Euclidean distance to compare:
1. $(AB)^{-1}$ 
2. $A^{-1}B^{-1}$
3. $B^{-1}A^{-1}$

Before starting to code, make a prediction about which results will be equal. Print out your results using formatting like the following:

```
Distance between (AB)^-1 and (A^-1)(B^-1) is ___
Distance between (AB)^-1 and (B^-1)(A^-1) is ___
Distance between (A^-1)(B^-1) and (B^-1)(A^-1) is ___
```

#### Grading: 
5 points per comparision. 